In [4]:
import os, sys
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()


## 1. Estructura de los Mensajes

La API usa una lista de mensajes con roles alternados `user` / `assistant`:

```python
messages = [
    {"role": "user",      "content": "Hola"},
    {"role": "assistant", "content": "Hola, ¿en qué puedo ayudarte?"},
    {"role": "user",      "content": "¿Cuál es la capital de Francia?"},
]
```

In [5]:
# Conversación manual: primer turno
conversation = []

def chat(user_message: str, model: str = DEFAULT_MODEL, max_tokens: int = 512) -> str:
    """Envía un mensaje y actualiza el historial de la conversación."""
    conversation.append({"role": "user", "content": user_message})
    
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=conversation
    )
    
    assistant_text = response.content[0].text
    conversation.append({"role": "assistant", "content": assistant_text})
    return assistant_text

# Primer mensaje
resp1 = chat("Mi nombre es Ana y estoy aprendiendo Python.")
print("Claude:", resp1)

Claude: ¡Hola Ana! 👋 

Me alegra saber que estás aprendiendo Python. Es un excelente lenguaje de programación para empezar, muy versátil y con una comunidad enorme.

¿En qué puedo ayudarte con tu aprendizaje de Python? Por ejemplo:

- **Conceptos básicos** (variables, tipos de datos, operadores)
- **Estructuras de control** (if/else, bucles)
- **Funciones**
- **Listas, diccionarios y otras estructuras de datos**
- **Programación orientada a objetos**
- **Librerías específicas**
- **Debugging o resolución de errores**
- **Ejercicios prácticos**

¿Hay algo en particular en lo que estés trabajando o que te gustaría aprender? 😊


In [6]:
# Segundo turno — Claude debe recordar el nombre
resp2 = chat("¿Recuerdas cómo me llamo?")
print("Claude:", resp2)

Claude: ¡Sí, claro! Te llamas **Ana** 😊

¿En qué te puedo ayudar con Python?


In [7]:
# Tercer turno
resp3 = chat("¿Qué lenguaje estaba aprendiendo?")
print("Claude:", resp3)

Claude: Estabas aprendiendo **Python** 🐍

¿Tienes alguna pregunta sobre Python o algún tema en particular que quieras repasar?


In [8]:
# Ver el historial completo
print("=== Historial de la conversacion ===")
for i, msg in enumerate(conversation):
    role = "Usuario" if msg["role"] == "user" else "Asistente"
    print(f"\n[{i+1}] {role}:")
    print(f"   {msg['content'][:150]}")


=== Historial de la conversacion ===

[1] Usuario:
   Mi nombre es Ana y estoy aprendiendo Python.

[2] Asistente:
   ¡Hola Ana! 👋 

Me alegra saber que estás aprendiendo Python. Es un excelente lenguaje de programación para empezar, muy versátil y con una comunidad e

[3] Usuario:
   ¿Recuerdas cómo me llamo?

[4] Asistente:
   ¡Sí, claro! Te llamas **Ana** 😊

¿En qué te puedo ayudar con Python?

[5] Usuario:
   ¿Qué lenguaje estaba aprendiendo?

[6] Asistente:
   Estabas aprendiendo **Python** 🐍

¿Tienes alguna pregunta sobre Python o algún tema en particular que quieras repasar?


## 2. Chat Interactivo en Terminal

El siguiente bloque implementa un chat interactivo en la consola. Escribe `salir` para terminar.

In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()

history_chat = []  # siempre reinicia al ejecutar la celda
print("Chat con Claude (escribe 'salir' para terminar)\n")
print("-" * 50)

while True:
    try:
        user_input = input("Tu: ")
    except EOFError:
        break

    if not user_input.strip() or user_input.strip().lower() in ("salir", "exit", "quit"):
        print("Hasta luego!")
        break

    history_chat.append({"role": "user", "content": user_input})

    response = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=1024,
        messages=history_chat
    )

    reply = response.content[0].text
    history_chat.append({"role": "assistant", "content": reply})
    print(f"\nClaude: {reply}\n")
    print("-" * 50)


## 3. Ejercicio: Chat con Memoria Limitada

**Tarea**: Implementa un chat interactivo que:
1. Acepte un parámetro `max_history` que limite cuántos turnos anteriores se envían a la API
2. Imprima cuántos tokens se usaron en cada respuesta
3. Permita conversar de forma interactiva (escribe `salir` para terminar)


In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()

def chat_with_memory(max_history: int = 5,
                     model: str = DEFAULT_MODEL) -> None:
    """
    Chat interactivo que envía solo los últimos `max_history` turnos a la API
    e imprime el uso de tokens en cada respuesta.
    """
    history = []  # siempre reinicia al llamar la función
    print(f"Chat con memoria limitada (max_history={max_history})")
    print("Escribe 'salir' para terminar\n")
    print("-" * 50)

    while True:
        try:
            user_input = input("Tu: ")
        except EOFError:
            break

        if not user_input.strip() or user_input.strip().lower() in ("salir", "exit", "quit"):
            print("Hasta luego!")
            break

        history.append({"role": "user", "content": user_input})

        # Limitar el historial enviado (mantener pares user/assistant)
        truncated = history[-(max_history * 2):]

        response = client.messages.create(
            model=model,
            max_tokens=512,
            messages=truncated
        )

        assistant_text = response.content[0].text
        history.append({"role": "assistant", "content": assistant_text})

        tokens = f"[tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out]"
        print(f"\nClaude: {assistant_text}\n")
        print(f"   {tokens}")
        print("-" * 50)


# Ejecutar el chat interactivo
chat_with_memory(max_history=5)
